# Perturbation Prediction with CellDISECT

This tutorial demonstrates how to use CellDISECT for **perturbation prediction** in single-cell data.  CellDISECT treats perturbations as a categorical covariate with **predefined external embeddings** (e.g. GenePT, ESM, scGPT) instead of learned embeddings.  This enables:

1. **Seen perturbation prediction** — predicting expression under a perturbation that was part of the training set.
2. **Unseen perturbation prediction** — predicting expression under a novel perturbation not seen during training, using its predefined embedding.
3. **Combinatorial perturbation prediction** — predicting expression under simultaneous perturbations (e.g. `GeneA+GeneB`), where the combined embedding is the sum of component embeddings.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

torch.set_float32_matmul_precision('medium')
scvi.settings.seed = 42

import warnings
warnings.filterwarnings('ignore')

from celldisect import CellDISECT, perturbation_metrics

## 1. Data Preparation

Load a perturbation dataset.  The AnnData object should have:

- `adata.layers['counts']` — raw count matrix
- `adata.obs['perturbation']` — perturbation labels (e.g. `'ctrl'`, `'GeneA'`, `'GeneA+GeneB'`)
- `adata.obs['cell_type']` — cell type annotations (or any other relevant covariates)

Replace the path below with your own data.

In [ ]:
adata = sc.read_h5ad('PATH/TO/PERTURBATION_DATA.h5ad')
adata = adata[adata.X.sum(1) != 0].copy()
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Perturbations: {adata.obs['perturbation'].nunique()} unique")
adata.obs['perturbation'].value_counts().head(10)

### 1.1 Load predefined gene embeddings

We need a dictionary mapping each **atomic** perturbation name to its embedding vector.  These embeddings can come from:

- **GenePT** — GPT-based gene embeddings
- **ESM** — protein language model embeddings
- **scGPT** — single-cell foundation model embeddings
- Any other source of gene/perturbation representations

The dictionary keys must include **every atomic perturbation** that appears in `adata.obs['perturbation']`.
For combinatorial perturbations like `'GeneA+GeneB'`, only the atomic components (`'GeneA'`, `'GeneB'`) need to be present — their combined embedding is computed automatically by summation.

In [ ]:
# Example: load GenePT embeddings
gene_embeddings = np.load('PATH/TO/GENE_EMBEDDINGS.npy', allow_pickle=True).item()

# Inspect
example_gene = list(gene_embeddings.keys())[0]
print(f"Number of gene embeddings: {len(gene_embeddings)}")
print(f"Example: '{example_gene}' -> shape {np.array(gene_embeddings[example_gene]).shape}")

In [ ]:
# Store embeddings in adata.uns
adata.uns['pert_embeddings'] = gene_embeddings

### 1.2 (Optional) Hold out a perturbation for evaluation

To evaluate unseen perturbation prediction, we hold out one perturbation from training and predict it later.

In [ ]:
holdout_pert = 'GENE_TO_HOLDOUT'  # Choose a perturbation to hold out

# Split: train on everything except the held-out perturbation
adata_train = adata[adata.obs['perturbation'] != holdout_pert].copy()
adata_test = adata.copy()  # Keep full data for evaluation

# The train adata still needs its own copy of embeddings
adata_train.uns['pert_embeddings'] = gene_embeddings
adata_test.uns['pert_embeddings'] = gene_embeddings

print(f"Training perturbations: {adata_train.obs['perturbation'].nunique()}")
print(f"Held-out perturbation: {holdout_pert}")

## 2. Model Setup and Training

In [ ]:
perturbation_key = 'perturbation'
cats = ['cell_type', perturbation_key]

CellDISECT.setup_anndata(
    adata_train,
    layer='counts',
    categorical_covariate_keys=cats,
    continuous_covariate_keys=[],
    perturbation_key=perturbation_key,
    perturbation_embedding_key='pert_embeddings',
    perturbation_combination_delimiter='+',
)

In [ ]:
model = CellDISECT(
    adata_train,
    n_layers=2,
    n_hidden=128,
    n_latent_shared=32,
    n_latent_attribute=32,
    dropout_rate=0.1,
)
print(model)

In [ ]:
model.train(
    max_epochs=200,
    batch_size=256,
    recon_weight=20,
    cf_weight=0.8,
    beta=0.003,
    clf_weight=0.05,
    adv_clf_weight=0.014,
    adv_period=5,
    n_cf=1,
    early_stopping=True,
    save_best=True,
    plan_kwargs={'lr': 0.003, 'weight_decay': 5e-5},
)

## 3. Predicting Seen Perturbations

First, let's predict a perturbation that the model has seen during training.

In [ ]:
seen_pert = 'SEEN_PERTURBATION'  # Replace with a perturbation from your training set

x_ctrl, x_true, x_pred = model.predict_perturbation(
    adata_test,
    perturbation=seen_pert,
    source_perturbation='ctrl',
    cats=cats,
    perturbation_key=perturbation_key,
    n_samples_from_source=500,
)

print(f"Control shape: {x_ctrl.shape}")
print(f"True shape: {x_true.shape}")
print(f"Predicted shape: {x_pred.shape}")

In [ ]:
metrics_seen = perturbation_metrics(
    x_pred.numpy(), x_true.numpy(), x_ctrl.numpy(), top_n_de=20
)
print(f"Metrics for seen perturbation '{seen_pert}':")
for k, v in metrics_seen.items():
    print(f"  {k}: {v:.4f}")

## 4. Predicting Unseen Perturbations

Now we predict the held-out perturbation that was **not** in the training data.
We pass its embedding via `new_embeddings`.

In [ ]:
x_ctrl, x_true, x_pred = model.predict_perturbation(
    adata_test,
    perturbation=holdout_pert,
    source_perturbation='ctrl',
    cats=cats,
    perturbation_key=perturbation_key,
    new_embeddings={holdout_pert: gene_embeddings[holdout_pert]},
    n_samples_from_source=500,
)

if x_true is not None:
    metrics_unseen = perturbation_metrics(
        x_pred.numpy(), x_true.numpy(), x_ctrl.numpy(), top_n_de=20
    )
    print(f"Metrics for unseen perturbation '{holdout_pert}':")
    for k, v in metrics_unseen.items():
        print(f"  {k}: {v:.4f}")
else:
    print(f"No ground-truth cells for '{holdout_pert}' — only prediction available.")
    print(f"Predicted shape: {x_pred.shape}")

## 5. Combinatorial Perturbation Prediction

CellDISECT handles combinatorial perturbations (e.g. `GeneA+GeneB`) by summing the atomic component embeddings.  The `+` delimiter is configurable.

In [ ]:
combo_pert = 'GENEA+GENEB'  # Replace with a valid combinatorial perturbation

x_ctrl, x_true, x_pred = model.predict_perturbation(
    adata_test,
    perturbation=combo_pert,
    source_perturbation='ctrl',
    cats=cats,
    perturbation_key=perturbation_key,
)

if x_true is not None:
    metrics_combo = perturbation_metrics(
        x_pred.numpy(), x_true.numpy(), x_ctrl.numpy(), top_n_de=20
    )
    print(f"Metrics for combinatorial perturbation '{combo_pert}':")
    for k, v in metrics_combo.items():
        print(f"  {k}: {v:.4f}")
else:
    print(f"No ground-truth for '{combo_pert}', predicted shape: {x_pred.shape}")

## 6. Evaluation & Visualization

In [ ]:
def plot_prediction_scatter(x_pred, x_true, x_ctrl, title='', top_n_de=20):
    """Scatter plot of predicted vs true mean expression."""
    pred_mean = x_pred.numpy().mean(0) if x_pred.ndim == 2 else x_pred.numpy()
    true_mean = x_true.numpy().mean(0) if x_true.ndim == 2 else x_true.numpy()
    ctrl_mean = x_ctrl.numpy().mean(0) if x_ctrl.ndim == 2 else x_ctrl.numpy()

    delta_true = true_mean - ctrl_mean
    de_idx = np.argsort(-np.abs(delta_true))[:top_n_de]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # All genes
    r_all, _ = pearsonr(pred_mean, true_mean)
    axes[0].scatter(true_mean, pred_mean, s=1, alpha=0.3)
    axes[0].set_xlabel('True mean expression')
    axes[0].set_ylabel('Predicted mean expression')
    axes[0].set_title(f'{title} — All genes (r={r_all:.3f})')
    lims = [min(axes[0].get_xlim()[0], axes[0].get_ylim()[0]),
            max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])]
    axes[0].plot(lims, lims, 'r--', alpha=0.5)

    # Top DE genes
    r_de, _ = pearsonr(pred_mean[de_idx], true_mean[de_idx])
    axes[1].scatter(true_mean[de_idx], pred_mean[de_idx], s=20, alpha=0.7, c='coral')
    axes[1].set_xlabel('True mean expression')
    axes[1].set_ylabel('Predicted mean expression')
    axes[1].set_title(f'{title} — Top {top_n_de} DE genes (r={r_de:.3f})')
    lims = [min(axes[1].get_xlim()[0], axes[1].get_ylim()[0]),
            max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])]
    axes[1].plot(lims, lims, 'r--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize seen perturbation prediction
# (Re-run predict_perturbation for seen_pert or use cached x_ctrl, x_true, x_pred)
plot_prediction_scatter(x_pred, x_true, x_ctrl, title=seen_pert)

### 6.1 Inspect perturbation embeddings

In [ ]:
names, emb_matrix = model.get_perturbation_embeddings()
print(f"Number of perturbation embeddings: {len(names)}")
print(f"Embedding matrix shape: {emb_matrix.shape}")
print(f"First 5 perturbation names: {names[:5]}")

### 6.2 Adding new perturbation embeddings at inference time

In [ ]:
# You can also add embeddings for new perturbations directly to the model
# (alternative to passing new_embeddings in predict_perturbation)
#
# model.add_perturbation_embedding('NewGene', gene_embeddings['NewGene'])
#
# After this, 'NewGene' will be available for prediction without new_embeddings.

## 7. Summary

In this tutorial we demonstrated:

- **Data preparation**: Storing predefined gene embeddings (GenePT/ESM) in `adata.uns` and setting up `setup_anndata` with `perturbation_key` and `perturbation_embedding_key`.
- **Model training**: Training CellDISECT with perturbation-aware embeddings.
- **Seen perturbation prediction**: Using `predict_perturbation` for perturbations in the training set.
- **Unseen perturbation prediction**: Providing `new_embeddings` for perturbations not seen during training.
- **Combinatorial perturbation prediction**: Predicting `GeneA+GeneB` via additive composition of embeddings.
- **Evaluation**: Using `perturbation_metrics` for Pearson correlation, MSE, and top-DE gene analysis.